# Checkout Friction Audit — Medallion Pipeline (GA4 Data API)
Bronze -> Silver -> Gold pipeline pulling checkout funnel events from
your own GA4 property and building daily friction KPIs.

In [0]:
%pip install google-analytics-data

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

## Config

In [0]:
SERVICE_ACCOUNT_PATH = "/Volumes/project/default/checkoutproject/fluent-aileron-500811-t0-f95dd83c0485.json"
PROPERTY_ID = "546475004"
FUNNEL_EVENTS = ["funnel_start", "view_item", "add_to_cart",
                 "begin_checkout", "add_payment_info", "purchase"]

BRONZE_TABLE = "checkout_friction.bronze_ga4_events"
SILVER_TABLE = "checkout_friction.silver_ga4_events"
GOLD_TABLE = "checkout_friction.gold_daily_friction"

# Date range to pull each run - adjust as needed, or parameterize via widgets
START_DATE = "2026-08-01"
END_DATE = "2026-08-07"

## Bronze Layer — Raw ingestion from GA4 Data API

In [0]:
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange, Dimension, Metric, RunReportRequest, Filter, FilterExpression
)
from google.oauth2 import service_account
import json
from datetime import datetime

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_PATH,
    scopes=["https://www.googleapis.com/auth/analytics.readonly"],
)
client = BetaAnalyticsDataClient(credentials=credentials)

request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[Dimension(name="date"), Dimension(name="eventName")],
    metrics=[Metric(name="eventCount")],
    date_ranges=[DateRange(start_date=START_DATE, end_date=END_DATE)],
    dimension_filter=FilterExpression(
        filter=Filter(
            field_name="eventName",
            in_list_filter=Filter.InListFilter(values=FUNNEL_EVENTS),
        )
    ),
)
response = client.run_report(request)

raw_rows = []
ingested_at = datetime.utcnow().isoformat()
for row in response.rows:
    raw_rows.append({
        "date_raw": row.dimension_values[0].value,   # YYYYMMDD
        "event_name": row.dimension_values[1].value,
        "event_count": int(row.metric_values[0].value),
        "ingested_at": ingested_at,
    })

print(f"Pulled {len(raw_rows)} raw rows from GA4 Data API")

Pulled 22 raw rows from GA4 Data API


/home/spark-22790648-1d19-40a5-ab5f-e7/.ipykernel/5282/command-7349709784806477-3993053061:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingested_at = datetime.utcnow().isoformat()


In [0]:
bronze_df = spark.createDataFrame(raw_rows)

spark.sql("CREATE SCHEMA IF NOT EXISTS checkout_friction")

(bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_TABLE))

display(spark.table(BRONZE_TABLE))

date_raw,event_count,event_name,ingested_at
20260806,1233,view_item,2026-08-08T18:27:55.905056
20260806,311,add_to_cart,2026-08-08T18:27:55.905056
20260803,242,view_item,2026-08-08T18:27:55.905056
20260804,242,view_item,2026-08-08T18:27:55.905056
20260806,130,begin_checkout,2026-08-08T18:27:55.905056
20260806,79,add_payment_info,2026-08-08T18:27:55.905056
20260806,58,purchase,2026-08-08T18:27:55.905056
20260803,56,add_to_cart,2026-08-08T18:27:55.905056
20260804,56,add_to_cart,2026-08-08T18:27:55.905056
20260807,45,view_item,2026-08-08T18:27:55.905056


## Silver Layer — Cleaned, deduplicated, typed

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table(BRONZE_TABLE)

silver_df = (
    bronze
    .withColumn("event_date", F.to_date("date_raw", "yyyyMMdd"))
    .withColumn("event_count", F.col("event_count").cast("long"))
    .filter(F.col("event_count").isNotNull())
    .filter(F.col("event_name").isin(FUNNEL_EVENTS))
)

# Deduplicate: keep the most recent ingestion per (date, event_name)
w = Window.partitionBy("event_date", "event_name").orderBy(F.col("ingested_at").desc())
silver_df = (
    silver_df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn", "date_raw")
)

(silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE))

display(spark.table(SILVER_TABLE).orderBy("event_date", "event_name"))

event_count,event_name,ingested_at,event_date
17,add_payment_info,2026-08-08T18:27:55.905056,2026-08-03
56,add_to_cart,2026-08-08T18:27:55.905056,2026-08-03
25,begin_checkout,2026-08-08T18:27:55.905056,2026-08-03
13,purchase,2026-08-08T18:27:55.905056,2026-08-03
242,view_item,2026-08-08T18:27:55.905056,2026-08-03
19,add_payment_info,2026-08-08T18:27:55.905056,2026-08-04
56,add_to_cart,2026-08-08T18:27:55.905056,2026-08-04
22,begin_checkout,2026-08-08T18:27:55.905056,2026-08-04
13,purchase,2026-08-08T18:27:55.905056,2026-08-04
242,view_item,2026-08-08T18:27:55.905056,2026-08-04


## Gold Layer — Daily funnel KPIs (pivoted, drop-off %, conversion %)

In [0]:
silver = spark.table(SILVER_TABLE)
 
pivoted = (
    silver.groupBy("event_date")
    .pivot("event_name", FUNNEL_EVENTS)
    .agg(F.sum("event_count"))
    .na.fill(0)
)
 
gold_df = (
    pivoted
    .withColumn("drop_1to2_pct", F.round(F.expr("try_divide((view_item - add_to_cart), view_item) * 100"), 2))
    .withColumn("drop_2to3_pct", F.round(F.expr("try_divide((add_to_cart - begin_checkout), add_to_cart) * 100"), 2))
    .withColumn("drop_3to4_pct", F.round(F.expr("try_divide((begin_checkout - add_payment_info), begin_checkout) * 100"), 2))
    .withColumn("drop_4to5_pct", F.round(F.expr("try_divide((add_payment_info - purchase), add_payment_info) * 100"), 2))
    .withColumn("overall_conversion_pct", F.round(F.expr("try_divide(purchase, view_item) * 100"), 2))
)
 
(gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE))
 
display(spark.table(GOLD_TABLE).orderBy("event_date"))

event_date,funnel_start,view_item,add_to_cart,begin_checkout,add_payment_info,purchase,drop_1to2_pct,drop_2to3_pct,drop_3to4_pct,drop_4to5_pct,overall_conversion_pct
2026-08-03,0,242,56,25,17,13,76.86,55.36,32.0,23.53,5.37
2026-08-04,0,242,56,22,19,13,76.86,60.71,13.64,31.58,5.37
2026-08-05,0,6,1,0,0,0,83.33,100.0,null,null,0.0
2026-08-06,0,1233,311,130,79,58,74.78,58.2,39.23,26.58,4.7
2026-08-07,0,45,10,8,7,6,77.78,20.0,12.5,14.29,13.33


## Quick sanity check — worst friction day

In [0]:
display(
    spark.table(GOLD_TABLE)
    .orderBy(F.col("drop_1to2_pct").desc())
    .limit(5)
)

event_date,funnel_start,view_item,add_to_cart,begin_checkout,add_payment_info,purchase,drop_1to2_pct,drop_2to3_pct,drop_3to4_pct,drop_4to5_pct,overall_conversion_pct
2026-08-05,0,6,1,0,0,0,83.33,100.0,null,null,0.0
2026-08-07,0,45,10,8,7,6,77.78,20.0,12.5,14.29,13.33
2026-08-04,0,242,56,22,19,13,76.86,60.71,13.64,31.58,5.37
2026-08-03,0,242,56,25,17,13,76.86,55.36,32.0,23.53,5.37
2026-08-06,0,1233,311,130,79,58,74.78,58.2,39.23,26.58,4.7
